# LAB 11 — Fine-tuning LLM bằng LoRA và QLoRA

**Nền tảng:** Google Colab GPU  
**Ngôn ngữ:** Python, PyTorch, Hugging Face Transformers, PEFT, TRL, bitsandbytes  
**Mô hình:** `Qwen/Qwen2.5-1.5B-Instruct`  
**Dữ liệu:** `bkai-foundation-models/vi-alpaca`

> Chọn **Runtime → Change runtime type → T4 GPU**, sau đó chạy lần lượt từ trên xuống hoặc chọn **Run all**.

Notebook không dùng TensorFlow nên không phụ thuộc `tensorflow-datasets` hay Protobuf của TensorFlow.

## 1. Mục tiêu học tập

Sau khi hoàn thành lab, sinh viên có thể:

1. Giải thích sự khác nhau giữa full fine-tuning, LoRA và QLoRA.
2. Chuẩn hóa dữ liệu instruction tuning thành định dạng hội thoại.
3. Fine-tune một LLM bằng LoRA 16-bit.
4. Fine-tune cùng LLM bằng QLoRA 4-bit NF4.
5. Đo số tham số huấn luyện, VRAM, thời gian, loss và kích thước adapter.
6. Lưu, tải lại, hợp nhất adapter và xây dựng demo hội thoại.
7. Nhận diện giới hạn của đánh giá LLM bằng loss và perplexity.

## 2. Kiến thức nền

### 2.1 Full fine-tuning

Full fine-tuning cập nhật toàn bộ tham số của mô hình:

\[
\theta' = \theta - \eta \nabla_\theta \mathcal{L}
\]

Cách này có khả năng thích nghi mạnh nhưng cần nhiều VRAM vì phải lưu trọng số, gradient và trạng thái optimizer cho toàn bộ mô hình.

### 2.2 LoRA

LoRA đóng băng trọng số gốc \(W\), sau đó học một cập nhật hạng thấp:

\[
W' = W + \Delta W,\qquad
\Delta W = \frac{\alpha}{r}BA
\]

Trong đó \(A\) và \(B\) nhỏ hơn rất nhiều so với \(W\). Chỉ các ma trận LoRA được cập nhật.

### 2.3 QLoRA

QLoRA giữ mô hình gốc ở dạng lượng tử hóa 4-bit, đồng thời huấn luyện adapter LoRA ở độ chính xác cao hơn:

\[
W_q = Q_{4bit}(W),\qquad
W' \approx W_q + \Delta W_{\text{LoRA}}
\]

Lab sử dụng:

- **NF4** cho trọng số 4-bit.
- **Double quantization** để giảm thêm bộ nhớ.
- **FP16/BF16 compute** cho phép tính.
- LoRA trên các lớp tuyến tính của Transformer.

## 3. LoRA và QLoRA khác nhau thế nào?

| Thuộc tính | LoRA | QLoRA |
|---|---|---|
| Trọng số mô hình gốc | FP16/BF16 | 4-bit |
| Tham số được cập nhật | Adapter LoRA | Adapter LoRA |
| VRAM | Thấp hơn full fine-tuning | Thấp nhất |
| Tốc độ | Thường nhanh hơn QLoRA | Có chi phí giải lượng tử |
| Chất lượng | Tốt | Thường gần LoRA/full fine-tuning |
| Hợp nhất adapter | Trực tiếp với base FP16 | Phải nạp lại base không lượng tử hóa |

## 4. Cài đặt thư viện

Các phiên bản được cố định để notebook có tính tái lập:

- Transformers 5.14.1
- TRL 1.8.0
- PEFT 0.19.1
- bitsandbytes 0.49.2
- Accelerate 1.14.0
- Datasets 5.0.0

Nếu vừa chạy một notebook khác có phiên bản Hugging Face cũ, hãy chọn **Runtime → Restart session** trước khi chạy cell này.

In [ ]:
import sys
import subprocess

packages = [
    "transformers==5.14.1",
    "trl==1.8.0",
    "peft==0.19.1",
    "bitsandbytes==0.49.2",
    "accelerate==1.14.0",
    "datasets==5.0.0",
    "sentencepiece>=0.2.0",
    "safetensors>=0.5.0",
    "gradio>=5.0,<7.0",
    "pandas>=2.0",
    "matplotlib>=3.8",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages]
)
print("Cài đặt hoàn tất.")

## 5. Import và kiểm tra môi trường

In [ ]:
import os
import gc
import math
import time
import json
import random
import shutil
import inspect
import warnings
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
)
from trl import SFTConfig, SFTTrainer

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "không xác định"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
for name in ["transformers", "trl", "peft", "bitsandbytes", "accelerate", "datasets"]:
    print(f"{name}: {package_version(name)}")

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "Không phát hiện GPU. Hãy chọn Runtime → Change runtime type → T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
gpu_vram_gb = gpu_props.total_memory / 1024**3
compute_capability = torch.cuda.get_device_capability(0)

bf16_supported = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_supported else torch.float16
use_tf32 = compute_capability[0] >= 8

torch.backends.cuda.matmul.allow_tf32 = use_tf32
torch.backends.cudnn.allow_tf32 = use_tf32

print("GPU:", gpu_name)
print(f"VRAM: {gpu_vram_gb:.2f} GB")
print("Compute capability:", compute_capability)
print("Compute dtype:", compute_dtype)
print("TF32:", use_tf32)

try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception:
    pass

## 6. Cấu hình thí nghiệm

`QUICK_MODE=True` phù hợp để kiểm tra toàn bộ notebook trên T4.

- LoRA và QLoRA đều chạy.
- Mỗi thí nghiệm giới hạn số bước để tránh chiếm GPU quá lâu.
- Chuyển sang `QUICK_MODE=False` sau khi bản nhanh chạy thành công.

In [ ]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "bkai-foundation-models/vi-alpaca"

QUICK_MODE = True
RUN_BASELINE = True
RUN_LORA = True
RUN_QLORA = True
RUN_GRADIO = False
MERGE_ADAPTER = False

if QUICK_MODE:
    TRAIN_SAMPLES = 800
    EVAL_SAMPLES = 80
    MAX_SEQ_LENGTH = 384
    MAX_STEPS = 100
    NUM_EPOCHS = 1
else:
    TRAIN_SAMPLES = 5000
    EVAL_SAMPLES = 300
    MAX_SEQ_LENGTH = 512
    MAX_STEPS = -1
    NUM_EPOCHS = 1

PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4

OUTPUT_ROOT = Path("/content/lab11_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    "Bạn là trợ lý AI hữu ích. Hãy trả lời bằng tiếng Việt rõ ràng, "
    "đúng trọng tâm và không bịa thông tin."
)

print({
    "model": MODEL_ID,
    "quick_mode": QUICK_MODE,
    "train_samples": TRAIN_SAMPLES,
    "eval_samples": EVAL_SAMPLES,
    "max_length": MAX_SEQ_LENGTH,
    "max_steps": MAX_STEPS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
})

## 7. Tải và khảo sát dữ liệu tiếng Việt

`vi-alpaca` chứa các trường instruction, input và output. Notebook có hàm chuẩn hóa linh hoạt để vẫn hoạt động khi tên trường thay đổi nhẹ.

Dữ liệu dự phòng chỉ dùng khi Hugging Face Hub không thể tải dataset.

In [ ]:
FALLBACK_DATA = [
    {
        "instruction": "Giải thích học máy là gì cho học sinh trung học.",
        "input": "",
        "output": "Học máy là cách giúp máy tính học quy luật từ dữ liệu để dự đoán hoặc ra quyết định mà không cần viết sẵn mọi quy tắc.",
    },
    {
        "instruction": "Viết ba lời khuyên để học lập trình hiệu quả.",
        "input": "",
        "output": "Hãy luyện tập mỗi ngày, tự làm dự án nhỏ và đọc kỹ thông báo lỗi thay vì chỉ sao chép lời giải.",
    },
    {
        "instruction": "Tóm tắt nội dung sau.",
        "input": "Năng lượng mặt trời là nguồn năng lượng tái tạo, sạch và ngày càng có chi phí thấp hơn.",
        "output": "Năng lượng mặt trời là nguồn năng lượng sạch, tái tạo và ngày càng kinh tế.",
    },
    {
        "instruction": "Phân biệt AI, machine learning và deep learning.",
        "input": "",
        "output": "AI là lĩnh vực rộng về máy móc thông minh; machine learning là nhánh cho máy học từ dữ liệu; deep learning là nhánh của machine learning dùng mạng nơ-ron nhiều lớp.",
    },
    {
        "instruction": "Viết email ngắn xin gia hạn bài tập một ngày.",
        "input": "Lý do: bị sốt.",
        "output": "Thưa thầy/cô, em đang bị sốt nên chưa thể hoàn thành bài đúng hạn. Em xin phép được gia hạn thêm một ngày và sẽ nộp bài sớm nhất có thể. Em cảm ơn thầy/cô.",
    },
    {
        "instruction": "Nêu hai ưu điểm của cơ sở dữ liệu quan hệ.",
        "input": "",
        "output": "Cơ sở dữ liệu quan hệ có cấu trúc rõ ràng và hỗ trợ truy vấn dữ liệu linh hoạt bằng SQL.",
    },
    {
        "instruction": "Giải thích overfitting.",
        "input": "",
        "output": "Overfitting xảy ra khi mô hình ghi nhớ dữ liệu huấn luyện quá mức nên hoạt động tốt trên dữ liệu cũ nhưng kém trên dữ liệu mới.",
    },
    {
        "instruction": "Đưa ra kế hoạch học tiếng Anh 30 phút mỗi ngày.",
        "input": "",
        "output": "Dành 10 phút học từ vựng, 10 phút nghe và nhắc lại, 5 phút đọc, rồi 5 phút viết vài câu sử dụng kiến thức vừa học.",
    },
    {
        "instruction": "Chuyển câu sau sang văn phong lịch sự.",
        "input": "Gửi tài liệu cho tôi ngay.",
        "output": "Bạn vui lòng gửi giúp tôi tài liệu khi thuận tiện nhé. Cảm ơn bạn.",
    },
    {
        "instruction": "Giải thích vì sao cần chia train, validation và test.",
        "input": "",
        "output": "Train dùng để học tham số, validation dùng để chọn cấu hình, còn test dùng để đánh giá cuối cùng trên dữ liệu chưa được sử dụng trong quá trình phát triển.",
    },
]

try:
    raw_dataset = load_dataset(DATASET_ID, split="train")
    print("Đã tải:", DATASET_ID)
except Exception as exc:
    print("Không tải được dataset từ Hub:", repr(exc))
    print("Sử dụng dữ liệu dự phòng.")
    raw_dataset = Dataset.from_list(FALLBACK_DATA)

print(raw_dataset)
print("Các cột:", raw_dataset.column_names)
print("Mẫu đầu tiên:")
print(raw_dataset[0])

## 8. Chuẩn hóa về prompt–completion

Với dữ liệu prompt–completion, `SFTTrainer` có thể chỉ tính loss trên phần completion. Điều này tránh bắt mô hình học lại toàn bộ system prompt và câu hỏi của người dùng.

In [ ]:
def first_text(example, candidates):
    for key in candidates:
        value = example.get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    return ""

def normalize_example(example):
    instruction = first_text(
        example, ["instruction", "prompt", "question", "query", "user"]
    )
    context = first_text(
        example, ["input", "context", "additional_input", "passage"]
    )
    answer = first_text(
        example, ["output", "response", "answer", "completion", "assistant"]
    )

    if context:
        user_content = f"{instruction}\n\nThông tin đầu vào:\n{context}"
    else:
        user_content = instruction

    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        "completion": [
            {"role": "assistant", "content": answer},
        ],
        "instruction_text": instruction,
        "input_text": context,
        "reference_text": answer,
    }

normalized = raw_dataset.map(
    normalize_example,
    remove_columns=raw_dataset.column_names,
    desc="Chuẩn hóa dữ liệu",
)

normalized = normalized.filter(
    lambda x: bool(x["instruction_text"].strip())
    and bool(x["reference_text"].strip()),
    desc="Loại mẫu rỗng",
)

needed = min(len(normalized), TRAIN_SAMPLES + EVAL_SAMPLES)
normalized = normalized.shuffle(seed=SEED).select(range(needed))

if len(normalized) < 2:
    raise ValueError("Dữ liệu hợp lệ quá ít để chia train/eval.")

eval_size = min(EVAL_SAMPLES, max(1, len(normalized) // 10))
split = normalized.train_test_split(test_size=eval_size, seed=SEED)

train_dataset = split["train"]
eval_dataset = split["test"]

print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print(train_dataset[0])

## 9. Tokenizer và kiểm tra chat template

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

sample_messages = (
    train_dataset[0]["prompt"] + train_dataset[0]["completion"]
)
formatted_sample = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False,
)

print(formatted_sample[:1500])
print("\nSố token:", len(tokenizer(formatted_sample)["input_ids"]))

## 10. Khảo sát độ dài chuỗi

Các mẫu vượt `MAX_SEQ_LENGTH` sẽ bị cắt. Tỷ lệ bị cắt quá cao là dấu hiệu cần tăng độ dài hoặc làm sạch dữ liệu.

In [ ]:
sample_for_length = train_dataset.select(
    range(min(300, len(train_dataset)))
)

def count_tokens(example):
    messages = example["prompt"] + example["completion"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"num_tokens": len(tokenizer(text, add_special_tokens=False)["input_ids"])}

length_dataset = sample_for_length.map(count_tokens)
lengths = np.array(length_dataset["num_tokens"])

print(pd.Series(lengths).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print(
    f"Tỷ lệ vượt {MAX_SEQ_LENGTH} token:",
    f"{(lengths > MAX_SEQ_LENGTH).mean():.2%}",
)

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(lengths, bins=30)
plt.axvline(MAX_SEQ_LENGTH, linestyle="--", label=f"max={MAX_SEQ_LENGTH}")
plt.xlabel("Số token")
plt.ylabel("Số mẫu")
plt.title("Phân bố độ dài mẫu huấn luyện")
plt.legend()
plt.show()

## 11. Hàm quản lý GPU và sinh văn bản

In [ ]:
def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def reset_peak_memory():
    cleanup_gpu()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

def gpu_memory_snapshot():
    return {
        "allocated_gb": torch.cuda.memory_allocated() / 1024**3,
        "reserved_gb": torch.cuda.memory_reserved() / 1024**3,
        "peak_allocated_gb": torch.cuda.max_memory_allocated() / 1024**3,
        "peak_reserved_gb": torch.cuda.max_memory_reserved() / 1024**3,
    }

def directory_size_mb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    total = sum(
        file.stat().st_size
        for file in path.rglob("*")
        if file.is_file()
    )
    return total / 1024**2

def parameter_stats(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "total_params": total,
        "trainable_params": trainable,
        "trainable_percent": 100 * trainable / total,
    }

@torch.inference_mode()
def generate_answer(model, tokenizer, prompt, max_new_tokens=160):
    model.eval()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.08,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    generated = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## 12. Bộ câu hỏi kiểm thử cố định

Các câu hỏi này không dùng để huấn luyện. Chúng giúp so sánh định tính giữa base model, LoRA và QLoRA.

In [ ]:
TEST_PROMPTS = [
    "Giải thích LoRA cho người mới học AI bằng một ví dụ dễ hiểu.",
    "Lập kế hoạch học Python trong 7 ngày, mỗi ngày 45 phút.",
    "Viết một đoạn phản hồi lịch sự khi khách hàng phàn nàn giao hàng chậm.",
    "Phân biệt precision và recall trong bài toán phân loại.",
    "Tóm tắt trong hai câu: Trí tuệ nhân tạo có thể hỗ trợ giáo dục bằng cách cá nhân hóa bài học, nhưng cần bảo vệ dữ liệu học sinh và tránh phụ thuộc quá mức vào công nghệ.",
]

pd.DataFrame({"prompt": TEST_PROMPTS})

## 13. Đánh giá base model trước fine-tuning

Base model được nạp 4-bit chỉ để tiết kiệm VRAM khi tạo mốc so sánh. Bước này không cập nhật tham số.

In [ ]:
def make_4bit_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_storage=torch.uint8,
    )

baseline_outputs = {}

if RUN_BASELINE:
    cleanup_gpu()
    baseline_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map={"": 0},
        quantization_config=make_4bit_config(),
        dtype=compute_dtype,
        low_cpu_mem_usage=True,
    )
    baseline_model.config.use_cache = True

    for prompt in TEST_PROMPTS:
        baseline_outputs[prompt] = generate_answer(
            baseline_model, tokenizer, prompt
        )

    display(
        pd.DataFrame(
            [
                {"prompt": prompt, "base_model": answer}
                for prompt, answer in baseline_outputs.items()
            ]
        )
    )

    del baseline_model
    cleanup_gpu()
else:
    print("Đã bỏ qua baseline.")

## 14. Cấu hình LoRA

`target_modules="all-linear"` áp dụng adapter lên các lớp tuyến tính của Transformer. Cùng cấu hình adapter được dùng cho cả hai thí nghiệm để phép so sánh tập trung vào FP16 và 4-bit.

In [ ]:
def make_lora_config():
    return LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
        use_rslora=False,
    )

print(make_lora_config())

## 15. Hàm xây dựng mô hình LoRA hoặc QLoRA

In [ ]:
def load_model_for_training(mode):
    mode = mode.lower()
    if mode not in {"lora", "qlora"}:
        raise ValueError("mode phải là 'lora' hoặc 'qlora'.")

    common_kwargs = {
        "pretrained_model_name_or_path": MODEL_ID,
        "device_map": {"": 0},
        "dtype": compute_dtype,
        "low_cpu_mem_usage": True,
    }

    if mode == "qlora":
        model = AutoModelForCausalLM.from_pretrained(
            **common_kwargs,
            quantization_config=make_4bit_config(),
        )
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(**common_kwargs)

    model.config.use_cache = False
    return model

## 16. Hàm tạo SFTConfig

- Batch thực là 1 để tránh tràn VRAM.
- Gradient accumulation tạo effective batch size bằng 8.
- `completion_only_loss=True` chỉ học trên phần trả lời.
- Không bật packing để tương thích ổn định với GPU T4.

In [ ]:
def make_training_args(mode, output_dir):
    optimizer = (
        "paged_adamw_8bit"
        if mode == "qlora"
        else "adamw_torch_fused"
    )

    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=NUM_EPOCHS,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.0,
        max_grad_norm=0.3,
        optim=optimizer,
        logging_strategy="steps",
        logging_steps=10,
        logging_first_step=True,
        eval_strategy="steps",
        eval_steps=50 if QUICK_MODE else 100,
        save_strategy="no",
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        fp16=(compute_dtype == torch.float16),
        bf16=(compute_dtype == torch.bfloat16),
        tf32=use_tf32,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        use_cache=False,
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        completion_only_loss=True,
        dataset_num_proc=min(2, os.cpu_count() or 1),
        dataloader_num_workers=min(2, os.cpu_count() or 1),
        dataloader_pin_memory=True,
        remove_unused_columns=True,
    )
    return SFTConfig(**kwargs)

## 17. Hàm chạy một thí nghiệm

Hàm này thực hiện đầy đủ:

1. Nạp mô hình.
2. Gắn adapter.
3. Huấn luyện.
4. Đánh giá loss/perplexity.
5. Sinh câu trả lời kiểm thử.
6. Lưu adapter.
7. Ghi nhận VRAM và thời gian.
8. Giải phóng GPU.

In [ ]:
experiment_results = []
generation_results = {}
training_logs = {}

def run_experiment(mode):
    mode = mode.lower()
    label = mode.upper()
    output_dir = OUTPUT_ROOT / f"{mode}_adapter"

    if output_dir.exists():
        shutil.rmtree(output_dir)

    cleanup_gpu()
    reset_peak_memory()

    load_start = time.perf_counter()
    model = load_model_for_training(mode)
    load_seconds = time.perf_counter() - load_start

    args = make_training_args(mode, output_dir)
    lora_config = make_lora_config()

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        peft_config=lora_config,
    )

    stats = parameter_stats(trainer.model)
    print(f"\n===== {label} =====")
    if hasattr(trainer.model, "print_trainable_parameters"):
        trainer.model.print_trainable_parameters()
    print(gpu_memory_snapshot())

    train_start = time.perf_counter()
    train_result = trainer.train()
    torch.cuda.synchronize()
    train_seconds = time.perf_counter() - train_start

    eval_metrics = trainer.evaluate()
    eval_loss = float(eval_metrics.get("eval_loss", float("nan")))
    perplexity = (
        math.exp(min(eval_loss, 20))
        if math.isfinite(eval_loss)
        else float("nan")
    )

    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    if hasattr(trainer.model, "gradient_checkpointing_disable"):
        trainer.model.gradient_checkpointing_disable()
    trainer.model.config.use_cache = True

    outputs = {}
    for prompt in TEST_PROMPTS:
        outputs[prompt] = generate_answer(
            trainer.model,
            tokenizer,
            prompt,
        )

    memory = gpu_memory_snapshot()
    logs = list(trainer.state.log_history)

    result = {
        "method": label,
        "base_precision": "4-bit NF4" if mode == "qlora" else str(compute_dtype).replace("torch.", ""),
        "train_samples": len(train_dataset),
        "max_steps": int(trainer.state.global_step),
        "trainable_params": stats["trainable_params"],
        "trainable_percent": stats["trainable_percent"],
        "load_seconds": load_seconds,
        "train_seconds": train_seconds,
        "train_loss": float(train_result.metrics.get("train_loss", float("nan"))),
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        "peak_vram_gb": memory["peak_reserved_gb"],
        "adapter_size_mb": directory_size_mb(output_dir),
        "adapter_path": str(output_dir),
    }

    del trainer
    del model
    cleanup_gpu()

    return result, outputs, logs

# PHẦN A — Fine-tuning bằng LoRA 16-bit

In [ ]:
if RUN_LORA:
    lora_result, lora_outputs, lora_logs = run_experiment("lora")
    experiment_results.append(lora_result)
    generation_results["LoRA"] = lora_outputs
    training_logs["LoRA"] = lora_logs
    display(pd.DataFrame([lora_result]))
else:
    print("Đã bỏ qua LoRA.")

# PHẦN B — Fine-tuning bằng QLoRA 4-bit

In [ ]:
if RUN_QLORA:
    qlora_result, qlora_outputs, qlora_logs = run_experiment("qlora")
    experiment_results.append(qlora_result)
    generation_results["QLoRA"] = qlora_outputs
    training_logs["QLoRA"] = qlora_logs
    display(pd.DataFrame([qlora_result]))
else:
    print("Đã bỏ qua QLoRA.")

## 18. So sánh kết quả định lượng

Perplexity được tính từ eval loss:

\[
\mathrm{PPL} = e^{\mathcal{L}_{eval}}
\]

Perplexity thấp hơn thường tốt hơn khi cùng tokenizer và cùng tập đánh giá. Nó không đảm bảo câu trả lời đúng sự thật hoặc hữu ích hơn.

In [ ]:
results_df = pd.DataFrame(experiment_results)

if len(results_df):
    numeric_columns = [
        "trainable_percent",
        "load_seconds",
        "train_seconds",
        "train_loss",
        "eval_loss",
        "perplexity",
        "peak_vram_gb",
        "adapter_size_mb",
    ]
    for column in numeric_columns:
        if column in results_df:
            results_df[column] = results_df[column].astype(float)

    display(
        results_df[
            [
                "method",
                "base_precision",
                "max_steps",
                "trainable_params",
                "trainable_percent",
                "train_seconds",
                "train_loss",
                "eval_loss",
                "perplexity",
                "peak_vram_gb",
                "adapter_size_mb",
            ]
        ].round(4)
    )
else:
    print("Chưa có thí nghiệm nào được chạy.")

In [ ]:
if len(results_df):
    plt.figure(figsize=(7, 4))
    plt.bar(results_df["method"], results_df["peak_vram_gb"])
    plt.ylabel("Peak VRAM reserved (GB)")
    plt.title("So sánh VRAM giữa LoRA và QLoRA")
    plt.show()

In [ ]:
if len(results_df):
    plt.figure(figsize=(7, 4))
    plt.bar(results_df["method"], results_df["train_seconds"])
    plt.ylabel("Thời gian huấn luyện (giây)")
    plt.title("So sánh thời gian huấn luyện")
    plt.show()

## 19. Đường loss trong quá trình huấn luyện

In [ ]:
def logs_to_dataframe(logs, method):
    rows = []
    for item in logs:
        if "loss" in item and "step" in item:
            rows.append(
                {
                    "method": method,
                    "step": item["step"],
                    "loss": item["loss"],
                }
            )
    return pd.DataFrame(rows)

loss_frames = [
    logs_to_dataframe(logs, method)
    for method, logs in training_logs.items()
]
loss_frames = [frame for frame in loss_frames if len(frame)]

if loss_frames:
    loss_df = pd.concat(loss_frames, ignore_index=True)
    plt.figure(figsize=(9, 4))
    for method, group in loss_df.groupby("method"):
        plt.plot(group["step"], group["loss"], marker="o", label=method)
    plt.xlabel("Step")
    plt.ylabel("Training loss")
    plt.title("Đường training loss")
    plt.legend()
    plt.show()
else:
    print("Không có log loss để vẽ.")

## 20. So sánh câu trả lời

Đánh giá mở cần kết hợp:

- Đúng nội dung.
- Làm theo yêu cầu.
- Rõ ràng và súc tích.
- Không bịa thông tin.
- Không tạo nội dung gây hại.

In [ ]:
comparison_rows = []

for prompt in TEST_PROMPTS:
    row = {"prompt": prompt}
    if baseline_outputs:
        row["Base"] = baseline_outputs.get(prompt, "")
    for method, outputs in generation_results.items():
        row[method] = outputs.get(prompt, "")
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

## 21. Bảng chấm điểm thủ công

Hãy tự cho điểm từng câu trả lời từ 1 đến 5:

| Tiêu chí | Mô tả |
|---|---|
| Correctness | Đúng nội dung |
| Relevance | Đúng yêu cầu |
| Clarity | Dễ hiểu |
| Conciseness | Không dài dòng |
| Safety | Không gây hại/bịa đặt |

Không dùng test prompts để tiếp tục điều chỉnh mô hình nếu muốn giữ phép đánh giá công bằng.

In [ ]:
manual_score_template = []

for index, prompt in enumerate(TEST_PROMPTS, start=1):
    for method in ["Base", "LoRA", "QLoRA"]:
        manual_score_template.append(
            {
                "prompt_id": index,
                "method": method,
                "correctness_1_5": None,
                "relevance_1_5": None,
                "clarity_1_5": None,
                "conciseness_1_5": None,
                "safety_1_5": None,
                "notes": "",
            }
        )

manual_scores_df = pd.DataFrame(manual_score_template)
display(manual_scores_df.head(9))

## 22. Tải lại adapter để suy luận

Adapter nhỏ không chứa toàn bộ mô hình. Khi sử dụng lại cần:

1. Nạp đúng base model.
2. Nạp adapter bằng `PeftModel.from_pretrained`.
3. Dùng đúng tokenizer.

In [ ]:
def load_adapter_for_inference(adapter_path, quantized=True):
    cleanup_gpu()

    load_kwargs = dict(
        pretrained_model_name_or_path=MODEL_ID,
        device_map={"": 0},
        dtype=compute_dtype,
        low_cpu_mem_usage=True,
    )

    if quantized:
        load_kwargs["quantization_config"] = make_4bit_config()

    base_model = AutoModelForCausalLM.from_pretrained(**load_kwargs)
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.config.use_cache = True
    model.eval()
    return model

available_adapters = {
    row["method"]: row["adapter_path"]
    for row in experiment_results
}

print(available_adapters)

### Ví dụ nạp adapter QLoRA

Cell dưới đây mặc định không chạy lại nếu adapter chưa tồn tại.

In [ ]:
demo_method = "QLORA" if "QLORA" in available_adapters else (
    next(iter(available_adapters)) if available_adapters else None
)

if demo_method:
    adapter_path = available_adapters[demo_method]
    reloaded_model = load_adapter_for_inference(
        adapter_path,
        quantized=True,
    )

    demo_prompt = "Nêu ba bước để bắt đầu một dự án học máy."
    print("Prompt:", demo_prompt)
    print(
        "Trả lời:",
        generate_answer(reloaded_model, tokenizer, demo_prompt),
    )

    del reloaded_model
    cleanup_gpu()
else:
    print("Chưa có adapter để tải lại.")

## 23. Hợp nhất adapter với base model — tùy chọn

- LoRA có thể được hợp nhất vào base FP16/BF16 bằng `merge_and_unload()`.
- Với QLoRA, không hợp nhất trực tiếp adapter vào base đang ở 4-bit. Phải nạp lại base không lượng tử hóa rồi mới hợp nhất.
- Mô hình đã hợp nhất lớn hơn adapter rất nhiều.

Cell này mặc định tắt vì có thể cần vài GB RAM, VRAM và dung lượng ổ đĩa.

In [ ]:
def merge_adapter(adapter_path, merged_output_dir):
    cleanup_gpu()

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map={"": 0},
        dtype=compute_dtype,
        low_cpu_mem_usage=True,
    )
    peft_model = PeftModel.from_pretrained(
        base_model,
        adapter_path,
    )
    merged_model = peft_model.merge_and_unload()
    merged_model.save_pretrained(
        merged_output_dir,
        safe_serialization=True,
    )
    tokenizer.save_pretrained(merged_output_dir)

    print("Đã lưu mô hình hợp nhất:", merged_output_dir)
    print(f"Kích thước: {directory_size_mb(merged_output_dir):.2f} MB")

    del merged_model
    del peft_model
    del base_model
    cleanup_gpu()

if MERGE_ADAPTER and available_adapters:
    selected_method = next(iter(available_adapters))
    merge_adapter(
        available_adapters[selected_method],
        OUTPUT_ROOT / f"{selected_method.lower()}_merged",
    )
else:
    print("MERGE_ADAPTER=False — bỏ qua bước hợp nhất.")

## 24. Lưu bảng kết quả và đóng gói adapter

In [ ]:
results_csv = OUTPUT_ROOT / "lora_qlora_benchmark.csv"
generations_csv = OUTPUT_ROOT / "generation_comparison.csv"
scores_csv = OUTPUT_ROOT / "manual_score_template.csv"

results_df.to_csv(results_csv, index=False)
comparison_df.to_csv(generations_csv, index=False)
manual_scores_df.to_csv(scores_csv, index=False)

archive_path = shutil.make_archive(
    "/content/LAB11_LoRA_QLoRA_Adapters",
    "zip",
    OUTPUT_ROOT,
)

print("Benchmark:", results_csv)
print("Generations:", generations_csv)
print("Score template:", scores_csv)
print("ZIP:", archive_path)

## 25. Đưa adapter lên Hugging Face Hub — tùy chọn

Không ghi token trực tiếp vào notebook. Dùng `notebook_login()` và chỉ push khi bạn hiểu rõ license của model và dataset.

In [ ]:
PUSH_TO_HUB = False
HUB_REPO_ID = "your-username/qwen2.5-1.5b-vietnamese-qlora"

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login

    notebook_login()

    selected = (
        available_adapters.get("QLORA")
        or available_adapters.get("LORA")
    )
    if not selected:
        raise RuntimeError("Chưa có adapter để push.")

    hub_model = load_adapter_for_inference(
        selected,
        quantized=True,
    )
    hub_model.push_to_hub(HUB_REPO_ID)
    tokenizer.push_to_hub(HUB_REPO_ID)

    del hub_model
    cleanup_gpu()
else:
    print("PUSH_TO_HUB=False — bỏ qua.")

## 26. Demo Gradio — tùy chọn

Đặt `RUN_GRADIO=True` ở phần cấu hình rồi chạy lại cell này. Demo tải adapter tốt nhất đang có ở dạng base 4-bit + adapter.

In [ ]:
if RUN_GRADIO and available_adapters:
    import gradio as gr

    selected_adapter = (
        available_adapters.get("QLORA")
        or available_adapters.get("LORA")
    )
    app_model = load_adapter_for_inference(
        selected_adapter,
        quantized=True,
    )

    def chat_fn(message, history):
        return generate_answer(
            app_model,
            tokenizer,
            message,
            max_new_tokens=200,
        )

    demo = gr.ChatInterface(
        fn=chat_fn,
        title="LAB 11 — Vietnamese LoRA/QLoRA Chatbot",
        description=f"Base model: {MODEL_ID}",
    )
    demo.launch(share=True, debug=False)
else:
    print("RUN_GRADIO=False hoặc chưa có adapter.")

## 27. Phân tích kết quả mong đợi

Thông thường:

- Số tham số trainable của LoRA và QLoRA gần như giống nhau vì dùng cùng adapter.
- Kích thước adapter gần như giống nhau.
- QLoRA dùng ít VRAM hơn rõ rệt vì trọng số base ở 4-bit.
- QLoRA không nhất thiết nhanh hơn LoRA vì phép tính cần giải lượng tử.
- Loss giảm không đồng nghĩa mô hình luôn trả lời đúng sự thật.
- Với chỉ 100 bước, thay đổi chất lượng có thể nhỏ. Đây là chế độ kiểm tra pipeline.

## 28. Các lỗi thường gặp

### CUDA out of memory

Giảm theo thứ tự:

1. `MAX_SEQ_LENGTH`: 384 → 256.
2. Giữ batch size bằng 1.
3. Tăng gradient accumulation nếu muốn giữ effective batch.
4. Chỉ chạy QLoRA.
5. Restart session để giải phóng hoàn toàn VRAM.

### `bitsandbytes` không phát hiện CUDA

- Kiểm tra đã chọn GPU.
- Restart session sau khi cài.
- Chạy `nvidia-smi`.
- Không sử dụng TPU.

### Dataset không tải được

- Kiểm tra Internet.
- Notebook tự dùng dữ liệu dự phòng.
- Có thể thay `DATASET_ID` bằng JSON/JSONL của bạn.

### Loss là NaN

- Giảm learning rate xuống `1e-4`.
- Kiểm tra dữ liệu rỗng hoặc quá dài.
- Thử `LORA_DROPOUT=0.1`.
- Không dùng dữ liệu có nội dung bị hỏng.

## 29. Bài tập thực hành

### Bài 1 — Thay đổi rank

Huấn luyện QLoRA với:

- \(r=8\)
- \(r=16\)
- \(r=32\)

So sánh số tham số trainable, adapter size, VRAM và eval loss.

### Bài 2 — Fine-tune theo miền

Tạo ít nhất 300 cặp instruction–response thuộc một miền:

- Tư vấn tuyển sinh
- Hỗ trợ khách hàng
- Hỏi đáp môn học
- Du lịch
- Công nghệ thông tin

Không đưa dữ liệu cá nhân, bí mật hoặc nội dung không có quyền sử dụng vào tập huấn luyện.

### Bài 3 — Đánh giá thủ công

Dùng tối thiểu 20 câu hỏi chưa xuất hiện trong train. Mỗi câu trả lời được hai người chấm độc lập.

### Bài 4 — So sánh độ dài chuỗi

Huấn luyện với 256 và 512 token. Phân tích trade-off giữa VRAM, tốc độ và chất lượng.

### Bài 5 — Triển khai

Lưu adapter lên Hugging Face Hub và xây một giao diện Gradio có system prompt tùy chỉnh.

## 30. Câu hỏi thảo luận

1. Vì sao QLoRA giảm VRAM nhưng không đảm bảo nhanh hơn LoRA?
2. Rank \(r\) ảnh hưởng thế nào đến số tham số và khả năng thích nghi?
3. Tại sao nên tính loss chủ yếu trên completion?
4. Adapter có thể hoạt động với một base model khác phiên bản không?
5. Vì sao eval loss không đủ để đánh giá chatbot?
6. Khi nào nên chọn full fine-tuning thay vì PEFT?
7. Những rủi ro nào xuất hiện khi fine-tune bằng dữ liệu tự thu thập?

## 31. Checklist nộp bài

- [ ] Notebook chạy từ đầu đến cuối trên GPU.
- [ ] Có bảng so sánh LoRA và QLoRA.
- [ ] Có biểu đồ VRAM và thời gian.
- [ ] Có ít nhất 10 prompt đánh giá riêng.
- [ ] Có nhận xét định tính.
- [ ] Có adapter đã lưu.
- [ ] Có mô tả nguồn và license dữ liệu.
- [ ] Không công khai token hoặc dữ liệu cá nhân.
- [ ] Có phần hạn chế và hướng cải tiến.

## 32. Tài liệu tham khảo

- Hugging Face PEFT — LoRA: https://huggingface.co/docs/peft/en/package_reference/lora
- Hugging Face PEFT — Quantization: https://huggingface.co/docs/peft/en/developer_guides/quantization
- Transformers — bitsandbytes: https://huggingface.co/docs/transformers/quantization/bitsandbytes
- TRL — SFTTrainer: https://huggingface.co/docs/trl/en/sft_trainer
- Qwen2.5-1.5B-Instruct: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
- Vietnamese Alpaca: https://huggingface.co/datasets/bkai-foundation-models/vi-alpaca
- LoRA paper: https://arxiv.org/abs/2106.09685
- QLoRA paper: https://arxiv.org/abs/2305.14314